# M2B Lab — Observable RAG with Evaluation in SageMaker

**Track:** Applied Agent Engineering Foundations  
**Module fit:** M2 — RAG Fundamentals & Retrieval Patterns  
**Derived from your upload:** stepwise RAG + observability + evaluation

## Learning goals
1. Load documents from an S3 prefix without writing back to S3
2. Chunk and embed them locally
3. Build a FAISS index
4. Retrieve evidence with scores
5. Generate an answer using Bedrock
6. Capture trace data in DataFrames
7. Run a lightweight evaluation loop


## Why this notebook matters

Your original stepwise RAG script already had strong observability ideas:
- retrieval scoring
- prompt capture
- evaluation
- run logs

This classroom version keeps that spirit, but makes it notebook friendly and SageMaker safe:
- local embeddings
- Bedrock generation
- DataFrame traces instead of backend JSON dependency
- optional local CSV export only


In [ ]:

# Uncomment only if packages are missing.
# %pip install -q boto3 pandas numpy sentence-transformers faiss-cpu python-docx pypdf rouge-score


## Step 1 — Configuration


In [ ]:

from __future__ import annotations

import io
import json
import os
import re
import time
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import faiss

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"

S3_BUCKET = "replace-me"
S3_PREFIX = "replace-me/rag_docs/"
USE_S3 = False

TOP_K = 3
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80

LOCAL_WORK_DIR = Path("./m2_observable_rag_artifacts")
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Region:", AWS_REGION)
print("Local artifacts:", LOCAL_WORK_DIR.resolve())


## Step 2 — S3 document loading helpers

This notebook reads:
- `.txt`
- `.md`
- `.csv`

For classroom stability we avoid formats that require heavier parsing stacks unless needed.


In [ ]:

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

def list_s3_keys(bucket: str, prefix: str) -> list[str]:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/") or key.split(".")[-1].lower() not in {"txt", "md", "csv"}:
                continue
            keys.append(key)
    return keys

def read_s3_text(bucket: str, key: str) -> str:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    if key.endswith(".csv"):
        df = pd.read_csv(io.BytesIO(body))
        return df.to_csv(index=False)
    return body.decode("utf-8", errors="ignore")

if USE_S3:
    doc_keys = list_s3_keys(S3_BUCKET, S3_PREFIX)
    raw_documents = [{"source": key, "text": read_s3_text(S3_BUCKET, key)} for key in doc_keys]
else:
    raw_documents = [
        {"source": "policy_1.txt", "text": "Employees must submit leave requests before planned absence. Emergency leave can be regularized later."},
        {"source": "policy_2.txt", "text": "Managers approve leave based on policy balance, business need, and notice period."},
        {"source": "policy_3.txt", "text": "Observability in enterprise RAG includes retrieval logs, prompts, model response traces, and evaluation metrics."},
        {"source": "policy_4.txt", "text": "Chunking strategy affects recall, grounding, and answer completeness in retrieval systems."},
    ]

docs_df = pd.DataFrame(raw_documents)
display(docs_df)


## Step 3 — Chunk documents


In [ ]:

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = text.strip()
    if not text:
        return []

    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)

    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step

    return chunks

chunk_rows = []
for row in raw_documents:
    for i, chunk in enumerate(chunk_text(row["text"]), start=1):
        chunk_rows.append({
            "source": row["source"],
            "chunk_id": f'{row["source"]}::chunk_{i:03d}',
            "chunk_text": chunk,
            "char_len": len(chunk)
        })

chunks_df = pd.DataFrame(chunk_rows)
display(chunks_df.head())
print("Total chunks:", len(chunks_df))


## Step 4 — Local embeddings + FAISS

We use a local sentence-transformer so the retrieval layer stays cheap and predictable during class.


In [ ]:

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(
    chunks_df["chunk_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False
)

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(np.array(chunk_embeddings, dtype="float32"))

print("Embedding dim:", embedding_dim)
print("FAISS vectors:", index.ntotal)


## Step 5 — Retrieval with trace capture


In [ ]:

def retrieve(query: str, top_k: int = TOP_K) -> tuple[pd.DataFrame, str]:
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), top_k)

    rows = []
    for rank, (score, idx_) in enumerate(zip(scores[0], idxs[0]), start=1):
        row = chunks_df.iloc[int(idx_)]
        rows.append({
            "query": query,
            "rank": rank,
            "score": float(score),
            "source": row["source"],
            "chunk_id": row["chunk_id"],
            "chunk_text": row["chunk_text"],
        })

    retrieval_df = pd.DataFrame(rows)
    context = "\n\n".join([f"[{r.rank}] {r.chunk_text}" for r in retrieval_df.itertuples()])
    return retrieval_df, context

test_retrieval_df, test_context = retrieve("Why does observability matter in enterprise RAG?")
display(test_retrieval_df)
print(test_context)


## Step 6 — Bedrock answer generation


In [ ]:

def call_bedrock_llm(prompt: str, model_id: str = BEDROCK_MODEL_ID, max_tokens: int = 250) -> str:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0.2}
    )
    return response["output"]["message"]["content"][0]["text"]

def answer_with_rag(query: str) -> tuple[pd.DataFrame, str, str]:
    retrieval_df, context = retrieve(query, top_k=TOP_K)
    prompt = f'''
You are an enterprise RAG assistant.
Answer only from the provided context.
If the answer is not present, say you do not know based on the context.

Context:
{context}

Question:
{query}

Answer:
'''.strip()

    answer = call_bedrock_llm(prompt)
    return retrieval_df, prompt, answer

query = "What should enterprise teams log in an observable RAG system?"
retrieval_df, prompt_used, answer = answer_with_rag(query)

display(retrieval_df)
print("\nPROMPT\n", prompt_used[:1200])
print("\nANSWER\n", answer)


## Step 7 — Trace tables instead of backend JSON

This is the classroom-safe observability pattern:
- keep run metadata in DataFrames
- optionally export to local CSV
- avoid a backend dependency during teaching


In [ ]:

run_metrics_df = pd.DataFrame([{
    "query": query,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "n_docs": len(docs_df),
    "n_chunks": len(chunks_df),
    "timestamp": pd.Timestamp.utcnow()
}])

display(run_metrics_df)


## Step 8 — Lightweight evaluation loop

Create a few golden questions and compare answer quality.
For simplicity, this notebook uses a token-overlap style score and exact evidence inspection.


In [ ]:

def normalize_text(s: str) -> list[str]:
    return re.findall(r"\w+", (s or "").lower())

def overlap_f1(pred: str, ref: str) -> float:
    p = normalize_text(pred)
    r = normalize_text(ref)
    if not p or not r:
        return 0.0
    p_set, r_set = set(p), set(r)
    inter = len(p_set & r_set)
    precision = inter / len(p_set) if p_set else 0.0
    recall = inter / len(r_set) if r_set else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

golden_df = pd.DataFrame([
    {
        "question": "Why does observability matter in enterprise RAG?",
        "reference_answer": "It helps teams inspect retrieval logs, prompts, responses, and evaluation metrics for debugging and governance."
    },
    {
        "question": "What affects answer completeness in retrieval systems?",
        "reference_answer": "Chunking strategy affects recall, grounding, and answer completeness."
    },
])

eval_rows = []
for row in golden_df.itertuples():
    retrieved, prmpt, pred = answer_with_rag(row.question)
    eval_rows.append({
        "question": row.question,
        "prediction": pred,
        "reference_answer": row.reference_answer,
        "overlap_f1": overlap_f1(pred, row.reference_answer),
        "top_source": retrieved.iloc[0]["source"] if len(retrieved) else None
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)
print("Average overlap_f1:", round(eval_df["overlap_f1"].mean(), 4))


## Step 9 — Optional local export


In [ ]:

retrieval_df.to_csv(LOCAL_WORK_DIR / "latest_retrieval.csv", index=False)
run_metrics_df.to_csv(LOCAL_WORK_DIR / "run_metrics.csv", index=False)
eval_df.to_csv(LOCAL_WORK_DIR / "evaluation.csv", index=False)

print("Saved local files:")
for p in sorted(LOCAL_WORK_DIR.glob("*")):
    print("-", p.name)


## Mini-hack — M2 retrieval tuning challenge

Give each learner pair one change:
1. Increase `TOP_K` and see whether answer quality improves or becomes noisy.
2. Reduce `CHUNK_SIZE` and inspect how retrieval changes.
3. Add one misleading document and see whether the system gets distracted.
4. Add a `latency_ms` column by timing retrieval and generation.

## Reflection questions
- What did observability make visible?
- Which retrieval traces are useful to show a platform team?
- When is a DataFrame log enough, and when do you need a full tracing stack?
